<a href="https://colab.research.google.com/github/VinayaSharada/KateelLearningDemosToStudents/blob/main/TreasuryAnalytics/P2PProcessMiningNotebook/p2p_process_mining_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P2P Process Mining Notebook

This notebook uses the same fictional Asteron Procure-to-Pay event log as the browser-based **P2P Process Mining Workbench**. The objective is not a tool tour. The objective is to show, in a transparent way, how finance can diagnose the actual AP process before deciding what to eliminate, standardize, enable, assure, and monitor.


## What we are trying to answer

How does a compliant invoice become payment-ready, what AP variants actually occur, where do queues and rework accumulate, and which issues should be fixed upstream before workflow automation begins?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW_CSV_URL = "https://raw.githubusercontent.com/VinayaSharada/KateelLearningDemosToStudents/main/TechUseCaseDemos/P2PProcessMiningWorkbench/Data/ap_event_log.csv"
LOCAL_CSV_CANDIDATES = [
    Path("TechUseCaseDemos/P2PProcessMiningWorkbench/Data/ap_event_log.csv"),
    Path("../TechUseCaseDemos/P2PProcessMiningWorkbench/Data/ap_event_log.csv"),
    Path("../../TechUseCaseDemos/P2PProcessMiningWorkbench/Data/ap_event_log.csv"),
]

def resolve_data_source():
    for candidate in LOCAL_CSV_CANDIDATES:
        if candidate.exists():
            return str(candidate)
    return RAW_CSV_URL

DATA_SOURCE = resolve_data_source()
TARGET_SLA_HOURS = 48
ACTIVE_TOUCH_HOURS = {
    "Invoice received": 0.3,
    "Three-way match": 1.2,
    "PO missing review": 1.4,
    "PO exception routed": 0.7,
    "PO confirmed": 0.6,
    "GRN chase": 0.9,
    "GRN posted": 0.5,
    "Duplicate review": 1.8,
    "Duplicate review reopened": 1.2,
    "Duplicate cleared": 1.0,
    "Approval requested": 0.2,
    "Approval escalated": 0.4,
    "Approved": 0.3,
    "Payment ready": 0.4,
    "Evidence completed": 0.6,
}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
print(f"Using data source: {DATA_SOURCE}")


### Why this setup cell matters

The notebook is designed to work in two places. In Colab it can use the raw GitHub URL after the repo is pushed. In a local repo checkout it should use the bundled fictional CSV first, which lets us validate the notebook before publishing.


In [ ]:
df = pd.read_csv(DATA_SOURCE, parse_dates=["timestamp"], keep_default_na=False)
df = df.sort_values(["case_id", "timestamp"]).reset_index(drop=True)
print(f"Rows: {len(df)}")
print(f"Cases: {df['case_id'].nunique()}")
df.head(10)


### How to interpret the raw event log

Each row is a timestamped business event for one invoice case. The event log does not directly tell us root cause. It tells us what happened, in what order, with what owner, and with what elapsed time. That is enough to begin a transformation discussion.


In [ ]:
happy_path = [
    "Invoice received",
    "Three-way match",
    "Approval requested",
    "Approved",
    "Payment ready",
]
print("Assumed happy path:")
print(" -> ".join(happy_path))
print(f"Assumed target SLA: {TARGET_SLA_HOURS} hours ({TARGET_SLA_HOURS/24:.1f} business days, illustrative)")


### Why we start with the happy path

The happy path is not the answer. It is the baseline assumption that many transformation discussions start with. The whole point of process mining is to test whether the executed path behaves like that assumption.


In [ ]:
cases = []
for case_id, group in df.groupby("case_id"):
    group = group.sort_values("timestamp").reset_index(drop=True)
    sequence = group["activity"].tolist()
    first = group.iloc[0]
    last = group.iloc[-1]
    lead_hours = (last["timestamp"] - first["timestamp"]).total_seconds() / 3600
    touch_hours = sum(ACTIVE_TOUCH_HOURS.get(activity, 0.5) for activity in sequence)
    wait_hours = max(0, lead_hours - touch_hours)
    repeated_count = len(sequence) - len(set(sequence))
    exception_rows = group[group["exception_type"] != "None"]
    exception_type = exception_rows.iloc[0]["exception_type"] if not exception_rows.empty else "None"
    exception_age_hours = ((last["timestamp"] - exception_rows.iloc[0]["timestamp"]).total_seconds() / 3600) if not exception_rows.empty else 0
    variant = " -> ".join(sequence)
    first_pass = exception_type == "None" and repeated_count == 0
    stp = variant == " -> ".join(happy_path)
    control_violation = ("Payment ready" in sequence and "Evidence completed" in sequence and sequence.index("Payment ready") < sequence.index("Evidence completed"))
    cases.append({
        "case_id": case_id,
        "entity": first["entity"],
        "amount_inr": first["amount_inr"],
        "variant": variant,
        "lead_hours": lead_hours,
        "touch_hours": touch_hours,
        "wait_hours": wait_hours,
        "rework_loops": repeated_count,
        "handoffs": len(sequence) - 1,
        "exception_type": exception_type,
        "exception_age_hours": exception_age_hours,
        "status": last["invoice_status"],
        "current_owner": last["owner"],
        "first_pass": first_pass,
        "stp": stp,
        "control_violation": control_violation,
    })

case_df = pd.DataFrame(cases)
case_df.head()


### What this case-level table gives us

This is where the event log becomes decision-ready. We reconstruct one row per invoice case and calculate lead time, touch time, queue time, rework, exception ageing, and conformance signals.


In [ ]:
variant_summary = (
    case_df.groupby("variant", as_index=False)
    .agg(
        cases=("case_id", "count"),
        avg_lead_hours=("lead_hours", "mean"),
        avg_wait_hours=("wait_hours", "mean"),
        avg_rework_loops=("rework_loops", "mean"),
    )
    .sort_values("cases", ascending=False)
)
variant_summary.head(8)


### How to read the variants

Each unique ordered activity sequence is a variant. This is the simplest way to show that the designed process is not the same as the executed process. A few recurring variants usually explain most of the operational friction.


In [ ]:
top_variants = variant_summary.head(6).copy()
plt.figure(figsize=(12, 5))
plt.barh(top_variants["variant"].str.slice(0, 70), top_variants["cases"], color="#0f766e")
plt.gca().invert_yaxis()
plt.xlabel("Cases")
plt.title("Top P2P Variants")
plt.tight_layout()
plt.show()


### Why this chart matters

When the distribution is concentrated in a few variants, the class can discuss where to intervene first. The visual is not the conclusion by itself. It is evidence that prioritization is possible.


In [ ]:
summary = pd.DataFrame({
    "metric": ["Cases", "Variant count", "Average lead time (hours)", "Average wait time (hours)", "Rework rate", "First-pass yield", "Straight-through-processing rate"],
    "value": [case_df["case_id"].nunique(), case_df["variant"].nunique(), round(case_df["lead_hours"].mean(), 1), round(case_df["wait_hours"].mean(), 1), f"{(case_df['rework_loops'].gt(0).mean()*100):.1f}%", f"{(case_df['first_pass'].mean()*100):.1f}%", f"{(case_df['stp'].mean()*100):.1f}%"]
})
summary


### How to interpret the core metrics

Lead time matters because it affects supplier experience, close discipline, and working-capital timing. Wait time matters because it usually signals policy ambiguity, upstream discipline gaps, or queue ownership problems rather than pure execution effort.


In [ ]:
friction_by_exception = (
    case_df.groupby("exception_type", as_index=False)
    .agg(cases=("case_id", "count"), avg_lead_hours=("lead_hours", "mean"), avg_wait_hours=("wait_hours", "mean"), avg_handoffs=("handoffs", "mean"), avg_rework_loops=("rework_loops", "mean"))
    .sort_values("avg_wait_hours", ascending=False)
)
friction_by_exception


### Why exception-level friction helps

Not every delay should be solved with workflow. Missing PO and missing GRN usually point to upstream discipline. Duplicate review may still require judgement. Delayed approvals may be good workflow candidates only after policy is stable.


In [ ]:
breaches = []
for _, row in case_df.iterrows():
    breach_list = []
    if row["control_violation"]:
        breach_list.append("Payment-ready status reached before evidence was complete")
    if row["exception_type"] == "Duplicate review" and row["exception_age_hours"] > 24:
        breach_list.append("Duplicate review unresolved beyond SLA")
    if row["exception_type"] != "None" and row["amount_inr"] > 500000 and row["wait_hours"] > 4:
        breach_list.append("High-value invoice waiting beyond assignment window")
    for breach in breach_list:
        breaches.append({
            "case_id": row["case_id"],
            "amount_inr": row["amount_inr"],
            "breach": breach,
            "age_hours": round(max(row["exception_age_hours"], row["wait_hours"]), 1),
            "current_owner": row["current_owner"],
            "evidence_status": "Completed after the fact" if row["control_violation"] else "Needs review",
        })

breach_df = pd.DataFrame(breaches).sort_values(["amount_inr", "age_hours"], ascending=[False, False])
breach_df.head(10)


### Why control exceptions deserve separate treatment

A control exception is not just another delay bucket. It crosses a governance boundary. The role of process mining here is to surface the behaviour clearly; the role of management is to decide what must remain controlled and human-reviewed.


In [ ]:
decision_board = pd.DataFrame([{"mined_insight": "Missing PO causes repeated rework", "decision_category": "Eliminate / standardize", "example_action": "Enforce PO discipline upstream and define a clear non-PO exception policy.", "expected_metric": "PO exception cycle time", "owner": "AP and procurement lead", "proof_point_30d": "20% lower PO exception turnaround"}, {"mined_insight": "Missing GRN cases create long queue time", "decision_category": "Redesign upstream process", "example_action": "Improve receiving discipline and GRN posting before automating downstream routing.", "expected_metric": "GRN ageing", "owner": "Operations receiving owner", "proof_point_30d": "Median GRN ageing below 24h"}, {"mined_insight": "Stable standard invoices can be routed automatically", "decision_category": "Enable workflow", "example_action": "Use low-code routing, reminders, and escalations only on the stable path.", "expected_metric": "Straight-through-processing rate", "owner": "P2P process owner", "proof_point_30d": "STP above 60% for four weeks"}, {"mined_insight": "Duplicate-review and policy exceptions still need judgement", "decision_category": "Keep human-reviewed", "example_action": "Provide evidence workbench and controller ownership instead of trying to automate away review judgement.", "expected_metric": "Duplicate-review queue ageing", "owner": "AP controller", "proof_point_30d": "Aged duplicate-review cases reduced by half"}, {"mined_insight": "Payment-ready control breaches need visible monitoring", "decision_category": "Assure / monitor", "example_action": "Log trigger, owner, evidence, and escalation deadline whenever the process crosses a control line.", "expected_metric": "Post-facto evidence completion", "owner": "Controller + internal controls", "proof_point_30d": "Zero new payment-ready-before-evidence cases"}])
decision_board


### What the decision board adds

The decision board prevents a common mistake: treating every mined problem as an automation use case. Some issues should be removed, some standardized, some automated, and some retained under explicit human judgement and control ownership.


In [ ]:
monitoring = pd.DataFrame([{"trigger": "High-value invoice exception unassigned > 4h", "hit": bool(((case_df["amount_inr"] > 500000) & (case_df["exception_type"] != "None") & (case_df["wait_hours"] > 4)).any()), "owner": "AP controller and budget owner", "evidence_logged": "Case owner, status, and ageing", "deadline": "4-hour escalation"}, {"trigger": "GRN missing > 24h", "hit": bool(((case_df["exception_type"] == "Missing GRN") & (case_df["exception_age_hours"] > 24)).any()), "owner": "Operations receiving owner", "evidence_logged": "Receiving note and GRN status", "deadline": "24-hour escalation"}, {"trigger": "Duplicate-review queue above five cases", "hit": bool((case_df["exception_type"] == "Duplicate review").sum() > 5), "owner": "AP controller", "evidence_logged": "Duplicate-review queue with reviewer", "deadline": "Same-day review"}, {"trigger": "STP rate below 60%", "hit": bool(case_df["stp"].mean() < 0.60), "owner": "P2P process owner", "evidence_logged": "Weekly STP trend", "deadline": "Weekly process review"}])
monitoring


### Why monitoring matters

Process mining should not be treated as a one-off diagnosis. Once the class chooses an intervention, it also needs a mechanism to see whether the process holds, drifts, or develops new exception patterns.


## What a strong classroom conclusion sounds like

A strong conclusion does **not** say: “The answer is automation.”

A strong conclusion says something like:

- the stable path is clear enough for workflow enablement,
- missing PO and missing GRN need upstream discipline first,
- duplicate review still requires human judgement,
- and control breaches need visible monitoring after launch.
